# Metabolic pathway maps

In [ ]:
# Setup. On Google Colab this downloads the course files and installs the dependencies (~2 min).
# Locally (conda env from environment.yml) it only moves to the course folder so `data/` is found.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/cell-factory-design-course"):
        !git clone -q --depth 1 https://github.com/lab-biotek-bio-ugm/cell-factory-design-course /content/cell-factory-design-course
        %pip install -q --progress-bar off -r /content/cell-factory-design-course/requirements.txt
    from google.colab import output
    output.enable_custom_widget_manager()  # needed for Escher maps

for folder in (".", "..", "/content/cell-factory-design-course"):
    if os.path.isdir(os.path.join(folder, "data")):
        os.chdir(folder)
        break

## Overview

**Learning objectives.** After this notebook you can:
1. Load a published Escher map and list the available maps.
2. Overlay reaction data (e.g. FBA fluxes) and metabolite data on a map.
3. Read an Escher map: arrow direction, color and width scales.

**Estimated time:** 30 min (10 min walkthrough, 15 min exercise).

**Prerequisites:** notebooks 01–02.

### Background

A flux vector from FBA has thousands of numbers, which is hard to inspect in a table. **Escher** ([King et al., 2015](https://doi.org/10.1371/journal.pcbi.1004321)) draws metabolic pathway maps in the browser: metabolites are circles (nodes), reactions are arrows (edges). Maps are drawn by hand for a specific model namespace (here BiGG identifiers), so reaction data given as `{reaction_id: value}` is matched to the arrows by ID. Reactions in the data that are not on the map are simply not shown.

When you pass `reaction_data`, Escher colors and scales the arrows by value; arrowheads follow the sign of the flux (a negative flux runs "backwards" relative to how the reaction is written). `metabolite_data` works the same way for the nodes (e.g. metabolomics concentrations). You can pan, zoom and hover over elements in the widget to see their IDs and values.

<div class="alert alert-info">

**Note:** Escher maps are interactive widgets. They need a running kernel (they do not show in a static preview of the notebook), and the maps are downloaded from the Escher website the first time you use them, so you need an internet connection.

</div>

## Metabolic pathway visualizations with Escher

In [ ]:
import escher

Display all available maps in Escher. Map names start with the model they were drawn for (e.g. `e_coli_core.`, `iJO1366.`).

In [ ]:
escher.list_available_maps()

Display a map of _E. coli_ central carbon metabolism.

In [ ]:
escher.Builder('e_coli_core.Core metabolism')

Visualize reaction-centric and/or metabolite-centric data. Here we give phosphoglycerate kinase (PGK) a value of 100 and ATP a value of 20, so only those two elements get colored.

In [ ]:
escher.Builder('e_coli_core.Core metabolism',
               reaction_data={'PGK': 100},
               metabolite_data={'ATP': 20})

## Exercise (15 min)

* Solve the _E. coli_ from the previous session and visualize the computed fluxes on the central carbon metabolism map.
* Visualize the same flux distribution on another map (a list of available maps can be retrieved by running `escher.list_available_maps()`)

### Solution

Solve the core model and pass the flux Series (converted to a dict) as `reaction_data`.

In [ ]:
from cobra.io import read_sbml_model

model = read_sbml_model("data/e_coli_core.xml.gz")
solution = model.optimize()
escher.Builder("e_coli_core.Core metabolism", reaction_data=solution.fluxes.to_dict())

The same fluxes on a larger map drawn for *i*JO1366. Reactions of the core model that also exist in *i*JO1366 (same BiGG ID) are colored; the rest of the map stays grey because the core model has no flux data for those reactions.

In [ ]:
escher.Builder("iJO1366.Central metabolism", reaction_data=solution.fluxes.to_dict())

## References

- King, Dräger, Ebrahim et al. (2015). Escher: A Web Application for Building, Sharing, and Embedding Data-Rich Visualizations of Biological Pathways. *PLOS Computational Biology* 11:e1004321. [doi.org/10.1371/journal.pcbi.1004321](https://doi.org/10.1371/journal.pcbi.1004321)